# V3 EDA 2 — 신규 단지: 누구이고, 분양권 · 분양가가 도움이 되나

## 목적

11 회귀에서 학습에 없던 단지(신규 단지)는 Test MAPE 13.6%로 가장 약하다. 설계서 Phase 4-3은 분양권 · 입주권 실거래와 청약홈 분양가로 이를 보완할 수 있는지 보기로 했다.
그 전에 **신규 단지 거래가 실제로 어떤 단지인지** 먼저 본다. 분양권 · 분양가는 막 입주한 신축에만 정보가 있다.

## 결론 요약

- **신규 단지 거래의 82%는 신축이 아니다.** 2023~25 중개거래 중 신규 단지 거래 2,500건(1.58%, 오차의 3.7%) 가운데 2,060건이 연식 10년 넘는 단지이고, 그중 89%가 100세대 미만이다. 데이터 기간에 처음 거래된 오래된 소규모 단지다.
  - 이 단지들은 공시가격 추정가가 99.8% 있는데도 MAPE 11.3%다. 단지 자체 거래 이력이 없는 것이 원인이고, 분양권 · 분양가와는 관계가 없다.
- **분양권 · 분양가가 정보를 줄 수 있는 신축(연식 2년 이하) 신규 단지 거래는 214건, 2023~25 중개거래의 0.13%다.**
  - 이 거래는 MAPE 15.0%로 약하고, 연식 0년은 실거래가가 예측보다 평균 15% 높다 (신축 첫 거래 과소추정).
  - 하지만 첫 거래가 학습에 들어간 뒤(기존 단지가 된 신축 3,053건)는 MAPE 5.87%로 전체 수준이다. 약한 구간은 입주 직후 몇 달뿐이다.

**판정:** 분양권 · 입주권 실거래와 분양가는 회귀 시험 후보에서 제외한다 (대상 0.13%, 분양권전매 API는 활용신청 필요). 신축 첫 거래 구간은 이미 등급 없이 참고용 범위만 보여주는 규칙(11 · 신규 단지)으로 다룬다.

## 원칙

- 매달 재학습 OOF(`models/regression/v2/oof_predictions.parquet`)의 2023~2025 중개거래만 본다. Test(2026)는 보지 않는다.
- `단지` = 그 달 모델 학습에 그 단지 거래가 있었는지 (기존 / 신규).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
trade = pd.read_parquet(PROJECT_DIR / "data/v2_5year_seoul/processed/v2_trade.parquet", columns=[
    "transaction_id", "단지일련번호", "단지명", "계약일자", "건축년도", "연식", "세대수", "거래유형", "거래금액", "공시기반_추정가", "예상가격_단지만"])
oof = pd.read_parquet(PROJECT_DIR / "models/regression/v2/oof_predictions.parquet", columns=["transaction_id", "ape", "r", "단지"])
d = trade.merge(oof, on="transaction_id", how="inner", validate="one_to_one")
d["연"] = pd.to_datetime(d["계약일자"]).dt.year
d = d[d["거래유형"].eq("중개거래") & d["연"].between(2023, 2025)].copy()
new = d[d["단지"].eq("신규")]
print(d["단지"].value_counts().to_dict())
print(f"신규 단지 거래 {len(new):,}건 ({len(new) / len(d):.2%}), 단지 {new['단지일련번호'].nunique():,}곳, MAPE {new['ape'].mean():.2f}% | "
      f"오차 비중 {new['ape'].sum() / d['ape'].sum():.1%}")

{'기존': 156023, '신규': 2500}
신규 단지 거래 2,500건 (1.58%), 단지 2,088곳, MAPE 11.23% | 오차 비중 3.7%


## 1. 신규 단지 거래의 연식 · 규모

In [2]:
agg = dict(거래=("ape", "size"), 단지=("단지일련번호", "nunique"), MAPE=("ape", "mean"), r평균=("r", "mean"))
age_bins = [-10, 0, 1, 2, 5, 10, 30, 100]
display(new.groupby(pd.cut(new["연식"], age_bins), observed=True).agg(**agg).assign(거래비중=lambda x: x["거래"] / len(new) * 100).round(3))
new.groupby(pd.cut(new["세대수"], [0, 100, 300, 1000, 1e5]), observed=True).agg(**agg).assign(거래비중=lambda x: x["거래"] / len(new) * 100).round(3)

,거래,단지,MAPE,r평균,거래비중
연식,,,,,
"(-10, 0]",67,44,19.565,0.150,2.68
"(0, 1]",86,45,12.287,-0.007,3.44
"(1, 2]",61,34,13.879,0.081,2.44
"(2, 5]",66,50,9.397,-0.004,2.64
"(5, 10]",158,63,5.470,-0.016,6.32
"(10, 30]",1925,1740,11.087,0.001,77.00
"(30, 100]",135,111,14.879,0.006,5.40


,거래,단지,MAPE,r평균,거래비중
세대수,,,,,
"(0.0, 100.0]",2094,1808,11.294,0.001,83.76
"(100.0, 300.0]",241,181,9.306,-0.008,9.64
"(300.0, 1000.0]",92,58,10.908,0.069,3.68
"(1000.0, 100000.0]",45,21,13.111,0.072,1.80


## 2. 처음 거래된 오래된 단지 (연식 10년 이상)

데이터 기간(2021-07~)에 처음 거래된 단지다. 단지 자체 이력(`예상가격_단지만`)은 없고, 공시가격 기반 추정가가 주된 정보다.

In [3]:
old_new = new[new["연식"] > 10]
print(f"연식 10년 넘는 신규 단지 거래 {len(old_new):,}건 ({len(old_new) / len(new):.1%}), 100세대 미만 {(old_new['세대수'] < 100).mean():.1%}")
print(f"공시기반_추정가 있음 {old_new['공시기반_추정가'].notna().mean():.1%} | 예상가격_단지만 있음 {old_new['예상가격_단지만'].notna().mean():.1%}")
has = old_new["공시기반_추정가"].notna()
print(f"MAPE: 공시 추정가 있음 {old_new.loc[has, 'ape'].mean():.2f}% (n={has.sum():,}) vs 없음 {old_new.loc[~has, 'ape'].mean():.2f}% (n={(~has).sum():,})")
old_new.sort_values("ape", ascending=False).drop_duplicates("단지일련번호").head(10)[["단지명", "건축년도", "세대수", "계약일자", "거래금액", "ape"]].round(1)

연식 10년 넘는 신규 단지 거래 2,060건 (82.4%), 100세대 미만 89.0%
공시기반_추정가 있음 99.8% | 예상가격_단지만 있음 2.8%
MAPE: 공시 추정가 있음 11.29% (n=2,055) vs 없음 30.20% (n=5)


,단지명,건축년도,세대수,계약일자,거래금액,ape
191004,오금그랑밸리,2002,12.0,2023-02-21,11000,172.1
26367,제이슨(360-7),2003,15.0,2024-05-17,17000,97.5
90086,Elleville,2000,10.0,2023-12-19,42500,77.2
180901,네오빌,2001,14.0,2024-01-31,720000,74.6
118220,양진마니아,2003,19.0,2024-01-11,40000,71.7
5940,삼각아파트C동,1970,46.0,2023-12-22,68213,68.9
206177,예건타워빌,2005,17.0,2023-03-14,21000,64.6
118350,탑건화곡동진선미,2006,18.0,2024-02-02,30000,62.7
170293,현대임페리움,2004,15.0,2023-09-21,100000,57.6
5507,빌라맨션,1974,71.0,2023-03-22,25000,56.5


## 3. 신축 (연식 2년 이하) — 분양권 · 분양가가 정보를 줄 수 있는 거래

신축 단지는 첫 거래가 학습에 들어가면 다음 달부터 기존 단지가 된다. 신축 전체를 단지 구분별로 본다.

In [4]:
young = d[d["연식"] <= 2]
display(young.groupby("단지").agg(**agg).assign(전체중비중=lambda x: x["거래"] / len(d) * 100).round(3))
yn = young[young["단지"].eq("신규")]
print(f"신축 · 신규 단지 거래 {len(yn):,}건 = 2023~25 중개거래의 {len(yn) / len(d):.2%} | 300세대 이상 {(yn['세대수'] >= 300).sum()}건")
print(f"연식 0년 신규 단지: r 평균 {new.loc[new['연식'] <= 0, 'r'].mean():+.3f} (실거래가 예측보다 높음 = 과소추정)")

,거래,단지,MAPE,r평균,전체중비중
단지,,,,,
기존,3053,177,5.873,0.027,1.926
신규,214,123,15.019,0.067,0.135


신축 · 신규 단지 거래 214건 = 2023~25 중개거래의 0.13% | 300세대 이상 96건
연식 0년 신규 단지: r 평균 +0.150 (실거래가 예측보다 높음 = 과소추정)


## 4. 분양권 · 입주권 실거래, 청약홈 분양가

- 국토부 「아파트 분양권전매 실거래가 자료」(`RTMSDataSvcSilvTrade`)는 현재 `DATA_GO_KR_API_KEY`로 호출하면 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR`(403)다 (2026-11-12, 활용신청 필요).
- 청약홈 분양정보(15098547)는 04에서 공고 · 세대수 · 입주예정월만 받았다. 주택형별 분양가(같은 서비스의 주택형별 상세)는 받지 않았다.
- 3장 결과로 보면 두 자료가 정보를 줄 수 있는 거래는 2023~25 중개거래의 0.1% 수준이어서, 활용신청 · 수집 · 단지 연결 작업을 하지 않기로 했다 (2026-11-12 결정).

## 5. 정리

| 항목 | 결과 | 다음 |
|---|---|---|
| 신규 단지 구성 | 82%가 처음 거래된 10년 넘은 단지(89%가 100세대 미만), 신축은 9% | 오래된 소규모 단지는 공시 추정가 외 추가 정보 없음 → 현재 규칙(등급 없음 · 참고용 범위) 유지 |
| 신축 첫 거래 | 214건(0.13%), MAPE 15.0%, 연식 0년 과소추정 15% | 분양권 · 분양가 수집 안 함. V4 화면: 입주 직후 단지는 "거래 이력 부족" 안내, 분양가 대비 표시는 표시값 후보 |
| 신축이 기존 단지가 된 뒤 | 3,053건 MAPE 5.87% | 처리 없음 |

V3 다음 단계는 EDA 3(단지 단위 1년 가격 변화가 잡음인지)이다.